# Two-model depth comparison viewer

Browse per-frame depth predictions and compare **any two models / runs** side by
side, frame by frame within a consecutive group.

Point it at your data by editing the **CONFIG** block in the setup cell:

- `MODEL_A` / `MODEL_B` — each `{label, dir}`, where `dir` is a run root
  containing `depth_npy/`. Labels drive every panel title, hover, and the diff.
- `RGB_DIR` — where the source images live (stem + `.jpg`/`.png`).
- `GROUP_RE` / `GROUP_SIZE` — how stems are bucketed into consecutive groups and
  how many frames each has (`GROUP_SIZE=None` accepts any length).
- `DEPTH_SCALE` — focal rescale (see below); set `1.0` to disable.

The default config compares the fine-tuned checkpoints **v62** vs **v63** on
`datasets/consecutive_groups` (groups of 11).

The main cell renders an interactive Plotly figure with three panels:
**RGB | model-A depth | model-B depth**. Hover any pixel to read depth in meters;
hovering the RGB reports *both* models at once. **Zooming or panning one panel
zooms all three** (linked axes). Use the **group** dropdown and **frame** slider
(+ **play**) to scrub through a group like a flip-book. Only groups present in
*both* runs appear, so it's safe to view while a run is still writing — re-run
the discovery cell to pick up newly-finished groups.

**Depth units.** For the DA3 runs, `run_inference.py` wrote metric depth assuming
a focal of 300 (`DATASET_FOCAL_AT_ORIG` / `CANONICAL_FOCAL`). The real-world
focal is 7200, so `DEPTH_SCALE = 7200/300 = 24` is applied to every prediction;
all depths, hovers, and stats are in **real-world meters**.

Works over Remote-SSH / port-forwarded Jupyter. Requires `ipywidgets`, `plotly`,
`anywidget`.

In [ ]:
import re
from pathlib import Path

import numpy as np
from PIL import Image as PILImage

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# ===========================================================================
# CONFIG — edit this block to compare ANY two models / runs.
# Each model is {label, dir}. `dir` is a run root that contains depth_npy/.
# Everything downstream (discovery, panels, hovers, titles, diff) is driven
# off MODEL_A / MODEL_B, so you only change these lines.
# ===========================================================================
RGB_DIR = REPO_ROOT / 'datasets' / 'consecutive_groups'

MODEL_A = {'label': 'v62',
           'dir': REPO_ROOT / 'workspace' / 'consecutive_v62_inference' / 'epoch_009'}
MODEL_B = {'label': 'v63',
           'dir': REPO_ROOT / 'workspace' / 'consecutive_v63_inference' / 'epoch_004'}

# Focal-length rescale. run_inference.py wrote metric depth assuming the
# dataset's true focal at original resolution is 300 (DATASET_FOCAL_AT_ORIG,
# CANONICAL_FOCAL). Metric depth from this model scales linearly with the
# assumed focal, so for the real-world focal of 7200 multiply by 7200/300 = 24.
# All depths / stats below are therefore in real-world meters. Set to 1.0 to
# disable rescaling for a dataset that doesn't need it.
ASSUMED_FOCAL = 300.0
REAL_FOCAL = 7200.0
DEPTH_SCALE = REAL_FOCAL / ASSUMED_FOCAL  # = 24.0

# How stems are grouped into a "consecutive group", and how many frames per
# group. GROUP_RE must capture the group id as group(1); set GROUP_SIZE=None to
# accept groups of any length (the slider adapts per group).
GROUP_RE = re.compile(r'^(g\d+)_')
GROUP_SIZE = 11
# ===========================================================================


def load_pred(run_dir: Path, stem: str) -> np.ndarray:
    """Load a prediction .npy for a stem from a run dir, in real-world meters."""
    return np.load(run_dir / 'depth_npy' / f'{stem}.npy').astype(np.float32) * DEPTH_SCALE


def rgb_path(stem: str) -> Path | None:
    for ext in ('.jpg', '.jpeg', '.png'):
        p = RGB_DIR / f'{stem}{ext}'
        if p.is_file():
            return p
    return None


print('RGB_DIR     :', RGB_DIR)
print(f"MODEL_A     : {MODEL_A['label']:<6} {MODEL_A['dir']}")
print(f"MODEL_B     : {MODEL_B['label']:<6} {MODEL_B['dir']}")
print(f'DEPTH_SCALE : {DEPTH_SCALE:g}  ({REAL_FOCAL:g} / {ASSUMED_FOCAL:g})')
assert RGB_DIR.is_dir(), f'not found: {RGB_DIR}'
for m in (MODEL_A, MODEL_B):
    assert (m['dir'] / 'depth_npy').is_dir(), f"no depth_npy/ under {m['dir']}"

## Discover consecutive groups

Group stems by `GROUP_RE` and keep only groups where **both** models have written
their depth `.npy` (and, if `GROUP_SIZE` is set, exactly that many frames).
Frames within a group are sorted by their trailing timestamp so the slider
scrubs in temporal order.

Re-run this cell as a still-running model finishes more groups to expand the list.

In [ ]:
def stems_in(run_dir: Path) -> set[str]:
    return {p.stem for p in (run_dir / 'depth_npy').glob('*.npy')}


a_stems = stems_in(MODEL_A['dir'])
b_stems = stems_in(MODEL_B['dir'])
common = a_stems & b_stems  # only stems both runs have predicted

# stem -> group id; group id -> sorted list of stems
groups: dict[str, list[str]] = {}
for stem in common:
    m = GROUP_RE.match(stem)
    if m:
        groups.setdefault(m.group(1), []).append(stem)


def frame_sort_key(stem: str):
    # stem looks like g001_000314_0001782081664978262 -> sort by trailing ts
    parts = stem.split('_')
    return int(parts[-1]) if parts[-1].isdigit() else stem


for gid in groups:
    groups[gid].sort(key=frame_sort_key)

# Keep complete groups. If GROUP_SIZE is set, require exactly that many frames
# in both runs; otherwise accept any group with >=1 frame.
if GROUP_SIZE is None:
    complete = {gid: fr for gid, fr in groups.items() if fr}
else:
    complete = {gid: fr for gid, fr in groups.items() if len(fr) == GROUP_SIZE}
GROUP_IDS = sorted(complete)

la, lb = MODEL_A['label'], MODEL_B['label']
print(f'{la} stems            : {len(a_stems)}')
print(f'{lb} stems            : {len(b_stems)}')
print(f'common stems         : {len(common)}')
req = 'any size' if GROUP_SIZE is None else f'size {GROUP_SIZE}'
print(f'complete groups ({req}): {len(GROUP_IDS)} / {len(groups)}')
assert GROUP_IDS, 'no complete groups found in both runs yet'
print(f'first / last group   : {GROUP_IDS[0]} .. {GROUP_IDS[-1]}')

## Frame-by-frame viewer: RGB | model-A | model-B

- **group** dropdown — pick a consecutive group.
- **frame** slider / **play** — scrub through that group like a flip-book (the
  slider range adapts to each group's frame count).

Three panels share pixel coordinates and **linked zoom/pan** — zoom or pan any
one panel and all three follow. Hover the RGB (via an invisible depth overlay)
to read *both* models' depth at that pixel, or hover either depth map for its own
value. The two depth panels share one colour scale (2/98 percentile of the
*union* of both preds for that frame) so A and B are comparable by colour, not
just by hover.

The title reports per-frame medians and the model-B − model-A median delta.

In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

LA, LB = MODEL_A['label'], MODEL_B['label']

# Invisible colorscale: put a transparent heatmap over the RGB so hover works
# there too (go.Image carries no interpolated depth values).
INVIS = [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']]


def frames_for(gid):
    return complete[gid]


group_dd = widgets.Dropdown(options=GROUP_IDS, value=GROUP_IDS[0], description='group')
_max0 = len(frames_for(GROUP_IDS[0])) - 1
frame_sl = widgets.IntSlider(min=0, max=_max0, step=1, value=0,
                             description='frame', continuous_update=False)
play = widgets.Play(min=0, max=_max0, step=1, interval=600, description='play')
widgets.jslink((play, 'value'), (frame_sl, 'value'))

fig = go.FigureWidget(
    make_subplots(
        rows=1, cols=3, horizontal_spacing=0.04,
        subplot_titles=('RGB', f'{LA} depth', f'{LB} depth'),
    )
)
# Trace order: 0 RGB image, 1 invisible-hover heatmap, 2 model-A depth,
# 3 model-B depth. The invisible overlay carries A in z and B in customdata so
# hovering the RGB reports BOTH models' depth at that pixel.
fig.add_trace(go.Image(z=np.zeros((2, 2, 3), dtype=np.uint8), hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Heatmap(
    z=[[0]], colorscale=INVIS, showscale=False,
    hovertemplate=('x=%{x} y=%{y}<br>'
                   f'{LA}=%{{z:.2f}} m<br>{LB}=%{{customdata:.2f}} m<extra></extra>')),
    row=1, col=1)
fig.add_trace(go.Heatmap(z=[[0]], colorscale='Inferno', showscale=False,
                         hovertemplate=f'x=%{{x}} y=%{{y}}<br>depth=%{{z:.2f}} m<extra>{LA}</extra>'),
              row=1, col=2)
fig.add_trace(go.Heatmap(z=[[0]], colorscale='Inferno',
                         colorbar=dict(title='depth (m)', x=1.005),
                         hovertemplate=f'x=%{{x}} y=%{{y}}<br>depth=%{{z:.2f}} m<extra>{LB}</extra>'),
              row=1, col=3)
fig.update_layout(height=440, width=1320, margin=dict(t=70, l=30, r=60, b=30))

# Linked zoom/pan: tie panels 2 & 3 to panel 1's axes so zooming one zooms all.
# xaxis/yaxis are col 1; x2/y2 col 2; x3/y3 col 3. `matches` shares the range;
# `scaleanchor` on each y keeps the per-panel square aspect.
fig.update_layout(
    xaxis2=dict(matches='x'), xaxis3=dict(matches='x'),
    yaxis2=dict(matches='y'), yaxis3=dict(matches='y'),
)


def render(gid: str, frame_idx: int):
    stem = frames_for(gid)[frame_idx]
    a = load_pred(MODEL_A['dir'], stem)
    b = load_pred(MODEL_B['dir'], stem)
    H, W = a.shape

    p = rgb_path(stem)
    rgb = (np.array(PILImage.open(p).convert('RGB').resize((W, H)))
           if p is not None else np.zeros((H, W, 3), dtype=np.uint8))

    # Shared colour scale from the union of both preds (fair comparison).
    both = np.concatenate([a.ravel(), b.ravel()])
    lo, hi = float(np.percentile(both, 2)), float(np.percentile(both, 98))

    with fig.batch_update():
        fig.data[0].z = rgb
        fig.data[1].z = a           # invisible overlay z: model-A depth over RGB
        fig.data[1].customdata = b  # ...and model-B depth for the same tooltip
        fig.data[2].z = a
        fig.data[2].zmin, fig.data[2].zmax = lo, hi
        fig.data[3].z = b
        fig.data[3].zmin, fig.data[3].zmax = lo, hi
        # Reset ranges to full frame on load; linked zoom then applies to all.
        for col in (1, 2, 3):
            fig.update_xaxes(range=[0, W], constrain='domain', row=1, col=col)
            ax = '' if col == 1 else str(col)
            fig.update_yaxes(range=[H, 0], scaleanchor=f'x{ax}', scaleratio=1,
                             row=1, col=col)
        da, db = float(np.median(a)), float(np.median(b))
        n = len(frames_for(gid))
        fig.layout.title = (f'{gid}  frame {frame_idx}/{n - 1}  |  {stem}<br>'
                            f'<sub>median  {LA}={da:.2f} m   {LB}={db:.2f} m   '
                            f'({LB}-{LA})={db - da:+.2f} m   scale x{DEPTH_SCALE:g}</sub>')


def _on_group(change):
    n = len(frames_for(change['new']))
    frame_sl.max = play.max = n - 1  # adapt to this group's frame count
    frame_sl.value = 0
    render(change['new'], 0)


def _on_frame(change):
    render(group_dd.value, change['new'])


group_dd.observe(_on_group, names='value')
frame_sl.observe(_on_frame, names='value')

render(group_dd.value, frame_sl.value)
display(widgets.VBox([widgets.HBox([group_dd, play, frame_sl]), fig]))

## Optional: model-B − model-A difference map for the current frame

Re-run this cell after moving the slider/dropdown above to render a signed
difference heatmap (`model-B − model-A`) for the frame currently selected. Red =
model-B predicts *farther*, blue = model-B predicts *nearer*. Hover reads the
per-pixel delta in meters.

In [ ]:
stem_cur = frames_for(group_dd.value)[frame_sl.value]
a = load_pred(MODEL_A['dir'], stem_cur)
b = load_pred(MODEL_B['dir'], stem_cur)
diff = b - a  # model-B minus model-A
H, W = diff.shape
lim = float(np.percentile(np.abs(diff), 98)) or 1e-6

dfig = go.Figure(go.Heatmap(
    z=diff, zmin=-lim, zmax=lim, colorscale='RdBu_r', zmid=0,
    colorbar=dict(title=f'{LB} - {LA} (m)'),
    hovertemplate='x=%{x} y=%{y}<br>Δ=%{z:+.2f} m<extra></extra>',
))
dfig.update_xaxes(range=[0, W], constrain='domain', title='x (px)')
dfig.update_yaxes(range=[H, 0], scaleanchor='x', scaleratio=1, title='y (px)')
dfig.update_layout(
    title=(f'{LB} - {LA}   {stem_cur}<br>'
           f'<sub>mean Δ={diff.mean():+.3f} m   |Δ| p98={lim:.2f} m   '
           f'red = {LB} farther, blue = {LB} nearer</sub>'),
    height=560, width=680, margin=dict(t=70, l=40, r=40, b=40),
)
dfig.show()